In [4]:
!nvidia-smi

Tue Nov  3 04:55:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
!pip install -q triton

In [6]:
import torch
import triton
import triton.language as t1

print("CUDA availabe:", torch.cuda.is_available())

if torch.cuda.is_available():
  print("GPU:", torch.cuda.get_device_name(0))
  print("Pytorch", torch.__version__)
  print("Triton:", triton.__version__)
else:
  raise RuntimeError("Enable GPU First")

CUDA availabe: True
GPU: Tesla T4
Pytorch 2.11.0+cu130
Triton: 3.6.0


In [13]:

import torch
import triton
import triton.language as tl


# ==========================================
# FLASH ATTENTION: SINGLE 3x3 TILE OPERATION
# ==========================================

@triton.jit
def flash_attention(Q, K, V, O):

    # Each program processes one query tile
    tile_id = tl.program_id(0)

    rows = tl.arange(0, 4)
    cols = tl.arange(0, 4)

    # --------------------------------------
    # 1. LOAD RANDOM Q TILE
    # --------------------------------------

    q = tl.load(
        Q + tile_id * 9 + rows[:, None] * 3 + cols[None, :],
        (rows[:, None] < 3) & (cols[None, :] < 3),
        other=0.0
    )

    # --------------------------------------
    # 2. LOAD FIXED K TILE FROM HBM
    # --------------------------------------

    k = tl.load(
        K + rows[:, None] * 3 + cols[None, :],
        (rows[:, None] < 3) & (cols[None, :] < 3),
        other=0.0
    )

    # --------------------------------------
    # 3. LOAD FIXED V TILE FROM HBM
    # --------------------------------------

    v = tl.load(
        V + rows[:, None] * 3 + cols[None, :],
        (rows[:, None] < 3) & (cols[None, :] < 3),
        other=0.0
    )

    # --------------------------------------
    # 4. CALCULATE QK^T
    # --------------------------------------

    scores = tl.sum(
        q[:, None, :] * k[None, :, :],
        axis=2
    )

    scores = scores * (1.0 / tl.sqrt(3.0))

    # Mask padded columns
    scores = tl.where(
        cols[None, :] < 3,
        scores,
        float("-inf")
    )

    # --------------------------------------
    # 5. SOFTMAX
    # --------------------------------------

    max_score = tl.max(scores, axis=1)

    exp_scores = tl.exp(
        scores - max_score[:, None]
    )

    denominator = tl.sum(exp_scores, axis=1)

    probabilities = (
        exp_scores / denominator[:, None]
    )

    # --------------------------------------
    # 6. CALCULATE ATTENTION OUTPUT
    # --------------------------------------

    output = tl.sum(
        probabilities[:, :, None] * v[None, :, :],
        axis=1
    )

    # --------------------------------------
    # 7. STORE OUTPUT TO HBM
    # --------------------------------------

    tl.store(
        O + tile_id * 9 + rows[:, None] * 3 + cols[None, :],
        output,
        (rows[:, None] < 3) & (cols[None, :] < 3)
    )


# ==========================================
# CREATE FIXED K AND V MATRICES
# ==========================================

K = torch.tensor(
    [
        [0.1, 0.2, 0.3],
        [0.4, 0.5, 0.6],
        [0.7, 0.8, 0.9]
    ],
    device="cuda",
    dtype=torch.float32
)

V = torch.tensor(
    [
        [1.0, 2.0, 3.0],
        [4.0, 5.0, 6.0],
        [7.0, 8.0, 9.0]
    ],
    device="cuda",
    dtype=torch.float32
)


# ==========================================
# GENERATE FOUR RANDOM Q MATRICES
# ==========================================

Q = torch.rand(
    (4, 3, 3),
    device="cuda",
    dtype=torch.float32
)

# Output for four attention tiles
O = torch.empty(
    (4, 3, 3),
    device="cuda",
    dtype=torch.float32
)


# ==========================================
# EXECUTE FOUR TILE OPERATIONS
# ==========================================

flash_attention[(4,)](Q, K, V, O)

torch.cuda.synchronize()


# ==========================================
# DISPLAY INPUTS AND OUTPUTS
# ==========================================

print("FIXED K MATRIX:\n", K.cpu())
print("\nFIXED V MATRIX:\n", V.cpu())

print("\n" + "=" * 45)

for i in range(4):

    print(f"\nQUERY TILE {i + 1}")
    print(Q[i].cpu())

    print(f"\nATTENTION OUTPUT {i + 1}")
    print(O[i].cpu())

    print("-" * 45)


# ==========================================
# LOGICAL KV TILE LOAD COUNT
# ==========================================

print("\nHBM -> ON-CHIP LOGICAL TILE LOADS")

print("Number of Q tiles:", 4)

print("K tile loads:", 4)
print("V tile loads:", 4)

print("Total K/V tile loads:", 4 + 4)


FIXED K MATRIX:
 tensor([[0.1000, 0.2000, 0.3000],
        [0.4000, 0.5000, 0.6000],
        [0.7000, 0.8000, 0.9000]])

FIXED V MATRIX:
 tensor([[1., 2., 3.],
        [4., 5., 6.],
        [7., 8., 9.]])


QUERY TILE 1
tensor([[0.6853, 0.2902, 0.5798],
        [0.2804, 0.6857, 0.9305],
        [0.0659, 0.8959, 0.7346]])

ATTENTION OUTPUT 1
tensor([[4.5324, 5.5324, 6.5324],
        [4.6455, 5.6455, 6.6455],
        [4.5794, 5.5794, 6.5794]])
---------------------------------------------

QUERY TILE 2
tensor([[0.7068, 0.5059, 0.4318],
        [0.1031, 0.9856, 0.7583],
        [0.8846, 0.6578, 0.1537]])

ATTENTION OUTPUT 2
tensor([[4.5621, 5.5621, 6.5621],
        [4.6291, 5.6291, 6.6291],
        [4.5793, 5.5793, 6.5792]])
---------------------------------------------

QUERY TILE 3
tensor([[0.4752, 0.6023, 0.0404],
        [0.0795, 0.1636, 0.4478],
        [0.9935, 0.6491, 0.0026]])

ATTENTION OUTPUT 3
tensor([[4.3848, 5.3848, 6.3848],
        [4.2388, 5.2388, 6.2388],
        [4.5623, 

In [14]:

import torch
import triton
import triton.language as tl


# -----------------------------------------
# GPU kernel: Load K only
# -----------------------------------------
@triton.jit
def load_k_kernel(K, OUT):
    x = tl.arange(0, 16)
    k = tl.load(K + x, x < 9, other=0.0)
    tl.store(OUT + x, k, x < 9)


# -----------------------------------------
# GPU kernel: Load V only
# -----------------------------------------
@triton.jit
def load_v_kernel(V, OUT):
    x = tl.arange(0, 16)
    v = tl.load(V + x, x < 9, other=0.0)
    tl.store(OUT + x, v, x < 9)


# -----------------------------------------
# Fixed 3x3 K and V in GPU memory
# -----------------------------------------
K = torch.tensor([
    [0.1, 0.2, 0.3],
    [0.4, 0.5, 0.6],
    [0.7, 0.8, 0.9]
], device="cuda", dtype=torch.float32)

V = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
    [7.0, 8.0, 9.0]
], device="cuda", dtype=torch.float32)

OUT_K = torch.empty(16, device="cuda")
OUT_V = torch.empty(16, device="cuda")


# -----------------------------------------
# Warm-up (exclude compilation time)
# -----------------------------------------
for _ in range(20):
    load_k_kernel[(1,)](K, OUT_K)
    load_v_kernel[(1,)](V, OUT_V)

torch.cuda.synchronize()


# -----------------------------------------
# Measure K loading kernel latency
# -----------------------------------------
repetitions = 1000

start_k = torch.cuda.Event(enable_timing=True)
end_k = torch.cuda.Event(enable_timing=True)

start_k.record()

for _ in range(repetitions):
    load_k_kernel[(1,)](K, OUT_K)

end_k.record()
torch.cuda.synchronize()

k_total_ms = start_k.elapsed_time(end_k)
k_avg_us = (k_total_ms * 1000) / repetitions


# -----------------------------------------
# Measure V loading kernel latency
# -----------------------------------------
start_v = torch.cuda.Event(enable_timing=True)
end_v = torch.cuda.Event(enable_timing=True)

start_v.record()

for _ in range(repetitions):
    load_v_kernel[(1,)](V, OUT_V)

end_v.record()
torch.cuda.synchronize()

v_total_ms = start_v.elapsed_time(end_v)
v_avg_us = (v_total_ms * 1000) / repetitions


# -----------------------------------------
# Results
# -----------------------------------------
print("=" * 45)
print("GPU TILE LOADING LATENCY")
print("=" * 45)

print(f"K loading total: {k_total_ms:.4f} ms")
print(f"K average kernel latency: {k_avg_us:.4f} us")

print()

print(f"V loading total: {v_total_ms:.4f} ms")
print(f"V average kernel latency: {v_avg_us:.4f} us")

print()
print(f"Combined average kernel latency: {k_avg_us + v_avg_us:.4f} us")
print("=" * 45)


GPU TILE LOADING LATENCY
K loading total: 34.8836 ms
K average kernel latency: 34.8836 us

V loading total: 21.6001 ms
V average kernel latency: 21.6001 us

Combined average kernel latency: 56.4837 us
